In [517]:
import os
import pandas as pd
import sqlite3
#  FOLDER PATH
folder_path = "D:/stock_project/ticker_csvs"
conn = sqlite3.connect("analysis.db")

In [518]:
all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_path, file))
        all_data.append(df)
df = pd.concat(all_data, ignore_index=True)
df.columns = df.columns.str.lower()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"])

MARKET SUMMARY

In [519]:
returns = []
for ticker in df["ticker"].unique():

    stock_df = df[df["ticker"] == ticker]

    first_close = stock_df.iloc[0]["close"]

    last_close = stock_df.iloc[-1]["close"]

    yearly_return = ((last_close - first_close)/ first_close) * 100

    returns.append([ticker,yearly_return])

returns_df = pd.DataFrame(returns,columns=["ticker", "Yearly_Return"])

print(returns_df.head())
#CALCULATING TOP GREEN STOCKS
top_green = returns_df.sort_values(by="Yearly_Return",ascending=False).head(10)
print("Top 10 Green Stocks:")
print(top_green)
top_green.to_sql("top_green_stocks",conn,if_exists="replace",index=False)
#CALCULATING TOP RED STOCKS
top_red = returns_df.sort_values(by="Yearly_Return").head(10)
print("Top 10 Loss Stocks:")
print(top_red )
top_red.to_sql("top_red_stocks",conn,if_exists="replace",index=False)

green_count = (returns_df["Yearly_Return"] > 0).sum()
red_count = (returns_df["Yearly_Return"] < 0).sum()
average_price = df["close"].mean()
average_volume = df["volume"].mean()
# MARKET SUMMARY
market_summary = pd.DataFrame({
    "green_stocks":[green_count],
    "red_stocks":[red_count],
    "average_price":[average_price],
    "average_volume":[average_volume]
})
print(market_summary)
market_summary.to_sql(
    "market_summary",
    conn,
    if_exists="replace",
    index=False
)

       ticker  Yearly_Return
0    ADANIENT       0.482569
1  ADANIPORTS      47.802626
2  APOLLOHOSP      44.585171
3  ASIANPAINT     -15.755397
4    AXISBANK      17.555052
Top 10 Green Stocks:
        ticker  Yearly_Return
47       TRENT     202.723364
8          BEL     112.122356
30         M&M     107.132545
5   BAJAJ-AUTO      77.414466
9   BHARTIARTL      71.799223
35   POWERGRID      67.678527
40   SUNPHARMA      60.840351
10        BPCL      60.184926
33        NTPC      57.115219
45       TECHM      55.315083
Top 10 Loss Stocks:
        ticker  Yearly_Return
24  INDUSINDBK     -30.322491
3   ASIANPAINT     -15.755397
7   BAJFINANCE     -10.545511
32   NESTLEIND      -5.863183
22  HINDUNILVR      -1.092123
0     ADANIENT       0.482569
6   BAJAJFINSV       1.780208
28   KOTAKBANK       2.148573
46       TITAN       4.263566
41  TATACONSUM       5.956998
   green_stocks  red_stocks  average_price  average_volume
0            45           5    2481.589733    6.997735e+06


1

  VOLATILITY ANALYSIS

In [520]:
#  Data cleaning & sorting
df["close"] = pd.to_numeric(df["close"], errors="coerce")

#  Daily return

df["daily_return"] = df.groupby("ticker")["close"].pct_change()

# Calculating Volatility 

volatility_df = df.groupby("ticker")["daily_return"].std().reset_index()
volatility_df.columns = ["ticker", "volatility"]
#Top10
top10_volatility = volatility_df.sort_values(
    by="volatility",
    ascending=False
).head(10)

print("VOLATILITY DATA\n", volatility_df)
print("\nTOP 10 VOLATILE STOCKS\n", top10_volatility)
volatility_df.to_sql("volatility_analysis", conn, if_exists="replace", index=False)
top10_volatility.to_sql("top10_volatility",conn, if_exists="replace", index=False)

VOLATILITY DATA
         ticker  volatility
0     ADANIENT    0.029218
1   ADANIPORTS    0.026583
2   APOLLOHOSP    0.014379
3   ASIANPAINT    0.012820
4     AXISBANK    0.015557
5   BAJAJ-AUTO    0.017550
6   BAJAJFINSV    0.013871
7   BAJFINANCE    0.015950
8          BEL    0.023817
9   BHARTIARTL    0.013821
10        BPCL    0.022545
11   BRITANNIA    0.013445
12       CIPLA    0.016533
13   COALINDIA    0.021675
14     DRREDDY    0.012642
15   EICHERMOT    0.016439
16      GRASIM    0.014558
17     HCLTECH    0.014423
18    HDFCBANK    0.013752
19    HDFCLIFE    0.014805
20  HEROMOTOCO    0.016588
21    HINDALCO    0.020056
22  HINDUNILVR    0.012348
23   ICICIBANK    0.013061
24  INDUSINDBK    0.019416
25        INFY    0.014641
26         ITC    0.012151
27    JSWSTEEL    0.016595
28   KOTAKBANK    0.014412
29          LT    0.017325
30         M&M    0.019533
31      MARUTI    0.013889
32   NESTLEIND    0.012229
33        NTPC    0.019898
34        ONGC    0.022920
35   POWERG

10

CUMULATIVE RETURN

In [523]:

#  Cumulative Return

df["cumulative_return"] = (
    (1 + df["daily_return"])
    .groupby(df["ticker"])
    .cumprod()
    - 1
)

#  Final cumulative return per stock
cum_history=df[["ticker","date","cumulative_return"]]
cum_df = df.groupby("ticker")["cumulative_return"].last().reset_index()
print("\nCUMULATIVE RETURN\n", cum_history.head(20))

top5 = cum_df.sort_values(
    by="cumulative_return",
    ascending=False
).head(5)

print("\nTOP 5 STOCKS\n", top5)
print(df.head())


cum_history.to_sql("cumulative_return", conn, if_exists="replace", index=False)
top5.to_sql("top5_cumulative_return", conn, if_exists="replace", index=False)


CUMULATIVE RETURN
       ticker                date  cumulative_return
0   ADANIENT 2023-11-01 05:30:00                NaN
1   ADANIENT 2023-11-02 05:30:00          -0.000902
2   ADANIENT 2023-11-03 05:30:00           0.005660
3   ADANIENT 2023-11-06 05:30:00           0.012944
4   ADANIENT 2023-11-07 05:30:00           0.007239
5   ADANIENT 2023-11-08 05:30:00           0.019235
6   ADANIENT 2023-11-09 05:30:00          -0.001195
7   ADANIENT 2023-11-10 05:30:00          -0.005502
8   ADANIENT 2023-11-12 05:30:00          -0.002683
9   ADANIENT 2023-11-13 05:30:00          -0.001646
10  ADANIENT 2023-11-15 05:30:00           0.003653
11  ADANIENT 2023-11-16 05:30:00          -0.005141
12  ADANIENT 2023-11-17 05:30:00          -0.003833
13  ADANIENT 2023-11-20 05:30:00          -0.030420
14  ADANIENT 2023-11-21 05:30:00          -0.009313
15  ADANIENT 2023-11-22 05:30:00          -0.020137
16  ADANIENT 2023-11-23 05:30:00          -0.018965
17  ADANIENT 2023-11-24 05:30:00           0

5

SECTOR PERFORMANCE

In [524]:
# sector File
sector_df = pd.read_csv("D:/stock_project/Sector_data.csv")

sector_df.columns = sector_df.columns.str.lower()
sector_df["symbol"] = sector_df["symbol"].str.split(":").str[-1].str.strip()

stock_tickers = set(df["ticker"].unique())
sector_tickers = set(sector_df["symbol"].unique())

missing = stock_tickers - sector_tickers
print(missing)

#Adding tickers
new_data = pd.DataFrame({
    "company": ["BRITANNIA","TATA CONSUMER","BHARTI AIRTEL","ADANI ENTERPRISES"],
    "sector": ["FMCG","FMCG","TELECOM","Energy"],
    "symbol": ["BRITANNIA","TATACONSUM","BHARTIARTL","ADANIENT"]
})

sector_df = pd.concat([sector_df, new_data], ignore_index=True)
sector_df = sector_df.drop_duplicates(subset="symbol")

sector_df = sector_df[["symbol", "sector"]]
sector_df.columns = ["ticker", "sector"]
#create dictionary
ticker_to_sector = dict(zip(sector_df["ticker"], sector_df["sector"]))

# Map sector

returns_df["sector"] = returns_df["ticker"].map(ticker_to_sector)

#  Sector performance

sector_perf = returns_df.groupby("sector")["Yearly_Return"].mean().reset_index()
print("\nSector Performance:\n", sector_perf)

sector_perf.to_sql("sector_performance", conn, if_exists="replace", index=False)


{'BHARTIARTL', 'BRITANNIA', 'ADANIENT', 'TATACONSUM'}

Sector Performance:
              sector  Yearly_Return
0         ALUMINIUM      40.933650
1       AUTOMOBILES      54.279084
2           BANKING      15.277737
3            CEMENT      35.719953
4           DEFENCE     112.122356
5            ENERGY      33.964058
6       ENGINEERING      24.460332
7            Energy       0.482569
8           FINANCE      13.686635
9              FMCG       5.040769
10   FOOD & TOBACCO       2.505223
11        INSURANCE      11.075406
12           MINING      35.023643
13    MISCELLANEOUS      46.193899
14           PAINTS     -15.755397
15  PHARMACEUTICALS      32.720604
16            POWER      62.396873
17        RETAILING     103.493465
18         SOFTWARE      44.816528
19            STEEL      28.681036
20          TELECOM      71.799223
21         TEXTILES      39.626038


22

STOCK PRICE CORRELATION

In [495]:
#Calculating correlation matrix
pivot_df = df.pivot(
    index="date",
    columns="ticker",
    values="close"
)

correlation_matrix = (pivot_df.corr())

print(correlation_matrix)
correlation_matrix.to_sql("correlation_matrix", conn, if_exists="replace", index=True, index_label="ticker")

correlation_matrix.index.name = "stock1"
correlation_matrix.columns.name = "stock2"
corr_long = (
    correlation_matrix
    .stack()
    .rename("correlation")
    .reset_index()
)
corr_long_pair = corr_long[
    (corr_long["stock1"] != corr_long["stock2"]) &
    (corr_long["stock1"] < corr_long["stock2"])
]
print(corr_long_pair.head())

#Top 10 Most Correlated Pairs
top_corr = (corr_long_pair.sort_values("correlation",ascending=False).head(10))
print(top_corr)
top_corr.to_sql("correlation_performance",conn,if_exists="replace",index=False)


ticker      ADANIENT  ADANIPORTS  APOLLOHOSP  ASIANPAINT  AXISBANK  \
ticker                                                               
ADANIENT    1.000000    0.791319    0.446468   -0.155700  0.403995   
ADANIPORTS  0.791319    1.000000    0.737667   -0.219709  0.702007   
APOLLOHOSP  0.446468    0.737667    1.000000   -0.115628  0.485768   
ASIANPAINT -0.155700   -0.219709   -0.115628    1.000000  0.059659   
AXISBANK    0.403995    0.702007    0.485768    0.059659  1.000000   
BAJAJ-AUTO  0.520872    0.850390    0.859841   -0.116982  0.674659   
BAJAJFINSV  0.015504    0.175439    0.528027    0.488847  0.314606   
BAJFINANCE -0.254531   -0.282005   -0.186790    0.644102  0.103988   
BEL         0.490427    0.880228    0.684202   -0.218585  0.831813   
BHARTIARTL  0.357043    0.784144    0.842401   -0.143534  0.719701   
BPCL        0.670278    0.914866    0.818285   -0.225451  0.571494   
BRITANNIA   0.313905    0.666778    0.709000    0.322980  0.754977   
CIPLA       0.548964

10

MONTHLY RETURN

In [496]:
# Monthly returns
monthly_returns = []

for (month, ticker), group in df.groupby(["month", "ticker"]):

    group = group.sort_values("date")

    first_close = group.iloc[0]["close"]
    last_close = group.iloc[-1]["close"]

    monthly_return = ((last_close - first_close)/ first_close) * 100

    monthly_returns.append([month,ticker,monthly_return])

monthly_returns_df = pd.DataFrame(
    monthly_returns,
    columns=[
        "month",
        "ticker",
        "monthly_return"
    ]
)
monthly_returns_df.to_sql("monthly_returns",conn,if_exists="replace",index=False)
print(monthly_returns_df.head(50))
conn.close()

      month      ticker  monthly_return
0   2023-11    ADANIENT        6.370360
1   2023-11  ADANIPORTS        7.333247
2   2023-11  APOLLOHOSP       15.269308
3   2023-11  ASIANPAINT        6.316130
4   2023-11    AXISBANK       10.542293
5   2023-11  BAJAJ-AUTO       13.962409
6   2023-11  BAJAJFINSV        6.399212
7   2023-11  BAJFINANCE       -4.684217
8   2023-11         BEL       10.196375
9   2023-11  BHARTIARTL       11.084351
10  2023-11        BPCL       22.079014
11  2023-11   BRITANNIA       10.355218
12  2023-11       CIPLA        0.919875
13  2023-11   COALINDIA       11.576716
14  2023-11     DRREDDY        8.261557
15  2023-11   EICHERMOT       18.726483
16  2023-11      GRASIM        7.326653
17  2023-11     HCLTECH        6.509393
18  2023-11    HDFCBANK        5.717192
19  2023-11    HDFCLIFE       11.402306
20  2023-11  HEROMOTOCO       23.495934
21  2023-11    HINDALCO       11.443700
22  2023-11  HINDUNILVR        2.964911
23  2023-11   ICICIBANK        2.286527
